In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib

In [2]:
def load_data_from_gsheet(sheet_id):
    data = pd.read_csv("CGWB_data_main_cleaned.csv")
    return data

In [3]:
def reshape_cgwb_data(df):
    meta_cols = ['Unnamed: 0', 'STATE', 'DISTRICT', 'LAT', 'LON', 'SITE_TYPE', 'WLCODE']
    id_vars = [c for c in df.columns if c in meta_cols]

    # Melt wide → long
    df_long = df.melt(
        id_vars=id_vars,
        var_name='TIME',
        value_name='GROUNDWATER'
    )

    # Convert TIME to datetime
    df_long['TIME'] = pd.to_datetime(df_long['TIME'], errors='coerce')
    df_long = df_long.dropna(subset=['TIME'])

    return df_long


In [4]:
def train_regressor_predict(data, lat, lon, target_year, tol=0.01, plot_results=True):
    # Filter site with tolerance
    group = data[
        (abs(data['LAT'] - lat) < tol) &
        (abs(data['LON'] - lon) < tol) &
        (data['GROUNDWATER'].notna()) &
        (data['GROUNDWATER'] != 0)
    ].sort_values(by='TIME')

    if group.empty:
        print(f"⚠️ No valid groundwater data near LAT: {lat}, LON: {lon}.")
        return None

    # Features: time → numeric (year, month), location
    group['Year'] = group['TIME'].dt.year
    group['Month'] = group['TIME'].dt.month

    X = group[['Year', 'Month', 'LAT', 'LON']]
    y = group['GROUNDWATER']

    # Train/Test Split
    split_index = int(len(X) * 0.8)
    X_train, y_train = X.iloc[:split_index], y.iloc[:split_index]
    X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]

    # Model
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"✅ Model Performance: RMSE={rmse:.2f}, R²={r2:.2f}")

    # Forecast Future
    last_date = group['TIME'].max()
    future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1),
                                 end=f"{target_year}-12-01", freq="MS")

    future_df = pd.DataFrame({
        'TIME': future_dates,
        'Year': future_dates.year,
        'Month': future_dates.month,
        'LAT': lat,
        'LON': lon
    })

    future_forecast = model.predict(future_df[['Year', 'Month', 'LAT', 'LON']])
    future_df['Forecast'] = future_forecast

    # Plot results
   # Plot results
    if plot_results:
        plt.figure(figsize=(12, 6))
        
        # Actual
        plt.plot(group['TIME'], y, label="Actual", color="blue")
        
        # Predictions (align with test indices)
        plt.plot(group.iloc[split_index:]['TIME'], y_pred,
                 label="Test Predictions", color="red")
        
        # Future forecast
        plt.plot(future_df['TIME'], future_df['Forecast'],
                 label="Future Forecast", color="green")
    
        plt.title(f"Groundwater Level Forecast (Regressor) - LAT: {lat}, LON: {lon}")
        plt.xlabel("Time")
        plt.ylabel("Groundwater Level")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        return {
        "model": model,
        "y_test": y_test,
        "y_pred": y_pred,
        "future_forecast": future_df
    }



In [5]:
if __name__ == "__main__":
    # Load CGWB dataset (local CSV)
    df = pd.read_csv("CGWB_data_wide.csv")

    # Reshape
    data = reshape_cgwb_data(df)

    # Inputs
    lat = float(input("Enter the latitude: "))
    lon = float(input("Enter the longitude: "))
    target_year = int(input("Enter the future year to forecast up to (e.g., 2028): "))

    results = train_regressor_predict(data, lat, lon, target_year)

    if results:
        forecast = results["future_forecast"]
        print("\n📅 Forecasted Yearly Average Groundwater Levels:\n")
        for year in sorted(forecast['Year'].unique()):
            yearly_vals = forecast[forecast['Year'] == year]['Forecast']
            print(f"{year}: {yearly_vals.mean():.2f}")

⚠️ No valid groundwater data near LAT: 78.45, LON: 28.75.
